In [ ]:
from unsloth import FastLanguageModel
import random

---

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-bnb-4bit/snapshots/70d37f62f4f475a5a42dbbd5b7aad38f420e0960"
local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Llama-3.1-8B-unsloth-bnb-4bit/snapshots/0a43e2563fa80ebc5eb093baa843c9cb65ab50d5"
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/llama-3-8b-bnb-4bit",
    model_name=local_model_path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # trust_remote_code=True, # Add this line
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

---

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        if len(instruction) == 0:
            augmented_instruction = ""
        else:
            augmented_instruction = "\n".join(instruction[random.randint(-len(instruction), -1):])
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(augmented_instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
# dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = load_dataset('json', data_files="../data/backup/dataset_alpaca_plain.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

# Split the dataset (e.g., 90% train, 10% validation)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42) # seed for reproducibility

# Access the splits
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"] # The 'test' split is commonly used for validation during training

In [ ]:
from datasets import load_dataset, Dataset
import json

# Load your custom dataset (replace with path if needed)
with open("../data/datasets/qa/qa_context.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convert each item to Unsloth-style format with custom prompt
# def format_example(example):
#     prompt = (
#         "Ты Леша. Напиши ответ на вопрос Сони. \n\n"
#         f"### Ваш предыдущий диалог:\n{example['context']}\n\n"
#         f"### Это новое сообщение:\n{example['instruction']}\n\n"
#         f"### Response:\n{example['response']}"
#     )
#     return { "text": prompt }


# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# Продолжи диалог. Ответь на вопрос Сони, как Леша. Прими во внимание контекст.

# ### Context:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# custom_prompt = """Ты Леша. Твоя задача максимально правдоподобно отвечать на вопросы Сони, принимая во внимание контекст диалога.
# ### Контекст диалога:
# {}

# ### Новое сообщение:
# {}

# ### Твой ответ:
# {}"""

# custom_prompt = """Ты Леша. Твоя задача максимально правдоподобно отвечать на вопросы Сони.
# ### Сообщение от Сони:
# {}

# ### Твой ответ:
# {}"""

custom_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Ты Леша - мой цифровой двойник.<|eot_id|><|start_header_id|>user<|end_header_id|>
Твоя задача максимально правдоподобно отвечать на вопросы Сони.

### Сообщение от Сони:
{}

### Твой ответ:
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

# EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(example):
    # instruction = "Продолжи диалог. Ответь на вопрос Сони, как Леша."
    context     = example['context']
    input       = example['instruction']
    output      = example['response']
    # texts = []
    # for instruction, input, output in zip(instructions, inputs, outputs):
    # Must add EOS_TOKEN, otherwise your generation will go on forever!
    # text = alpaca_prompt.format(context, input, output)
    # text = custom_prompt.format(context, input, output)
    text = custom_prompt.format(input, output)
    # texts.append(text)
    return { "text" : text, }

# Apply formatting
# formatted_data = [format_example(example) for example in raw_data]
formatted_data = [formatting_prompts_func(example) for example in raw_data]

# Create HuggingFace dataset
dataset = Dataset.from_list(formatted_data)

In [ ]:
dataset[0]

---

In [ ]:
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        do_eval = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 170,
        # max_steps = 60,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        eval_strategy = "steps",      # Added evaluation
        eval_steps = 100,                   # Evaluate every 100 steps
        save_strategy = "steps",
        save_steps = 100,
        load_best_model_at_end = True,
        save_total_limit = 2,
        # report_to="wandb", # Optional: Integrate Weights & Biases for better tracking
    ),
)

In [ ]:
trainer_stats = trainer.train()

---

## Inference

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
messages = [
    "Соня: <сообщение>",
    "Леша: <сообщение>",
    "Соня: <сообщение>",
]
inputs = tokenizer(
[
    # alpaca_prompt.format(
    #     "Continue the fibonnaci sequence.", # instruction
    #     "1, 1, 2, 3, 5, 8", # input
    #     "", # output - leave this blank for generation!
    # )
    alpaca_prompt.format(
        "\n".join(messages), # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 64,
    top_p=0.95,
    temperature=0.9,
    # repetition_penalty=1.2,
    use_cache = True,
    # do_sample=True
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Format the prompt using the *inference* template
context = ""  # Or provide actual dialogue context if available
input_message = "Ты меня любишь?"
# prompt_text = inference_prompt_template.format(
prompt_text = custom_prompt.format(
    # context,
    input_message,
    ""
)

inputs = tokenizer(
    [prompt_text],
    return_tensors = "pt"
).to("cuda")


# EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
outputs = model.generate(
    **inputs,
    max_new_tokens = 128,
    # top_p=0.9,
    # temperature=0.7,
    repetition_penalty=1.2,
    use_cache = True,
    # do_sample=True,
) # Tell the model when to stop generating)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
tokenizer.decode(outputs[0])